# Introducción a la inyección de fallos por voltaje (Voltage Glitching)

**RESUMEN:** *Aunque no es tan sofisticado como el hardware de glitching del ChipWhisperer Lite o del ChipWhisperer Pro, el ChipWhisperer Nano también es capaz de realizar inyecciones de fallos. En esta práctica, realizaremos algunas pruebas sencillas de glitching sobre la placa objetivo del Nano, mostrando cómo explorar el rango de parámetros de configuración y observando el efecto que tienen sobre el hardware.*

**OBJETIVOS DE APRENDIZAJE:**

* Comprender cómo se puede emplear el glitching de tensión para alterar el funcionamiento de un dispositivo objetivo.
* Explorar los parámetros de configuración de glitching para determinar aquellos que resultan efectivos.

Los dispositivos de hardware digital tienen ciertos requisitos de tensión y reloj para funcionar correctamente. Si estos requisitos no se cumplen, el dispositivo puede dejar de funcionar o incluso resultar dañado. Al cortocircuitar los pines de tensión de un microcontrolador durante periodos breves y controlados, **podemos hacer que se comporte de forma errática, borrando registros u omitiendo instrucciones**. Estos ataques pueden ser enormemente potentes en la práctica. Consideremos, por ejemplo, el siguiente código procedente de `linux-util-2.24`:

```C
/*
 *   auth.c -- PAM authorization code, common between chsh and chfn
 *   (c) 2012 by Cody Maloney <cmaloney@theoreticalchaos.com>
 *
 *   this program is free software.  you can redistribute it and
 *   modify it under the terms of the gnu general public license.
 *   there is no warranty.
 *
 */

#include "auth.h"
#include "pamfail.h"

int auth_pam(const char *service_name, uid_t uid, const char *username)
{
    if (uid != 0) {
        pam_handle_t *pamh = NULL;
        struct pam_conv conv = { misc_conv, NULL };
        int retcode;

        retcode = pam_start(service_name, username, &conv, &pamh);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_authenticate(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_acct_mgmt(pamh, 0);
        if (retcode == PAM_NEW_AUTHTOK_REQD)
            retcode =
                pam_chauthtok(pamh, PAM_CHANGE_EXPIRED_AUTHTOK);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_setcred(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        pam_end(pamh, 0);
        /* no need to establish a session; this isn't a
         * session-oriented activity...  */
    }
    return TRUE;
}
```

Este es el código de inicio de sesión (*login*) del sistema operativo Linux. Observa que si pudiéramos omitir la comprobación de `if (uid != 0)` y simplemente saltar al final, evitaríamos tener que introducir una contraseña. Este es el verdadero poder de los ataques por inyección de fallos (*glitch attacks*): no se trata de romper el cifrado, sino sencillamente de eludir por completo el módulo de autenticación.

### Hardware de Glitching

La configuración de glitching del ChipWhisperer Nano es bastante sencilla. Al igual que sus hermanos mayores, el Lite y el Pro, utiliza un MOSFET para cortocircuitar la alimentación de tensión del microcontrolador a masa (GND):

<img src="img/Glitch-vccglitcher.png" alt="Crowbar" width="300"/>

En el caso del Nano, la señal `Glitch In` está controlada por 2 parámetros:

1. `scope.glitch.ext_offset`: El fallo se insertará aproximadamente a los `8.3ns * scope.glitch.ext_offset`
2. `scope.glitch.repeat`: El fallo se mantendrá insertado durante aproximadamente `8.3ns * scope.glitch.repeat`

<img src="img/glitch.png" alt="Crowbar" width="700"/>

Durante esta práctica, iremos variando estos parámetros para comprobar si logramos que el objetivo cometa un error en el cálculo que esté ejecutando.

Como siempre, lo primero que hacemos es definir nuestra plataforma hardware:

In [ ]:
SCOPETYPE = 'CWNANO'
PLATFORM = 'CWNANO'
SS_VER = 'SS_VER_2_1'

A continuación realizamos la configuración básica del hardware. Al finalizar esta celda, los objetos `scope` (que controla la inyección de fallos y medición) y `target` (que controla la comunicación serie) quedan creados y conectados.

In [ ]:
%run "../chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

La siguiente celda compila el firmware (código que va a ejecutar) que se va a cargar en el target (`simpleserial-glitch.c`). 

El código que va a ejecutar la víctima es una variante del `simpleserial` que básicamente recibe comandos por puerto serie (de un carácter), algunos además de un buffer, y genera una respuesta que envía también por el puerto serie. 

En esta práctica se le va a enviar el comando `g` (`target.simpleserial_write("g", bytearray([]))`), que si inspeccionas la función correspondiente en `simpleserial-glitch.c` asociada a dicho comando es similar a esta:

```C
uint8_t glitch_loop(uint8_t cmd, uint8_t scmd, uint8_t len, uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;

    trigger_high(); // Le dice al ChipWhisperer: "¡Empieza el ataque ahora!"

    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }

    trigger_low(); // Fin de la ventana de ataque

    simpleserial_put('r', 4, (uint8_t*)&cnt); // Envía el resultado de vuelta
    return (cnt != 2500) ? 0x10 : 0x00;
}
```

Como se puede observar, el comando `g` debe retornar siempre el valor 2500, que se calcula en el bucle anidado de 50 x 50 iteraciones.

Observa también que justo antes de comenzar el bucle donde se calcula dicho valor, se activa el trigger para avisar de que ahí debe comenzar el ataque. 


In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd Firmwares/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

Una vez compilado el firmware, lo grabamos en el target:

In [ ]:
fw_path = "Firmwares/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

A continuación, definimos la frecuencia de reloj del target (7.5 MHz) y definimos la función `reboot_flush()` que se encarga de reiniciar por hardware el target (haciendo reset con `scope.io.nrst`) y limpiando el buffer del puerto serie, ya que en nuestro ataque el microcontrolador target se colgará por completo muchas veces.

In [ ]:
scope.io.clkout = 7.5E6
def reboot_flush():            
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.nrst = "high_z"
    time.sleep(0.05)
    #Flush garbage too
    target.flush()

Antes de continuar, podemos ver el estado actual del osciloscopio, para ver las configuraciones que hemos hecho.

In [ ]:
scope

Realicemos ahora una ejecución "limpia" (sin ataque) para ver que todo funciona correctamente:
1. Primero hacemos un reinicio del target (`reboot_flush()`) para partir de un estado limpio.
2. Armamos el sistema (`scope.arm()`) que se queda a la espera de la señal de inicio (`trigger`).
3. Enviamos el comando `g` al target para que empiece el cómputo (que dará como resultado 2500) con `target.simpleserial_write()`
4. Mientras, el sistema atacante se queda a la espera de que termine la ejecución (`scope.capture()`).
5. Una vez terminada, se lee la respuesta por el puerto serie con `target.simpleserial_read_witherrors()` que devuelve un diccionario con varios campos, entre ellos `valid` que indica si la respuesta es correcta (no ha habido reseteo ni cuelgue del target). En `payload` debería estar el valor calculado por el target (2500) pero en su representación hexadecimal (0x000009C4).

Ejecuta la celda y comprueba que se ejecuta correctamente. 

In [ ]:
reboot_flush()
scope.arm()
target.simpleserial_write("g", bytearray([]))
scope.capture()
val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']
print(val)

Para realizar un ataque, bastaría con configurar para el objeto `scope` los parámetros del glitch (`ext_offset` y `repeat`) antes del envio del comando `g`. En el siguiente bloque definimos una función que hace el ataque con los parámetros que se le pasen. Si te fijas es muy similar al bloque anterior, solo que previamente define `scope.glitch.repeat` y `scope.glitch.ext_offset`.

In [ ]:
import struct
def insert_glitch(offset, repeat):
    """
    Inyecta un glitch y devuelve:
        - int: el contador 'cnt' devuelto por el target (ej. 2500 o corrupto).
        - None: si el target se colgó o se reinició.
    """
    # configuramos los parámetros del glitch  
    scope.glitch.repeat = repeat
    scope.glitch.ext_offset = offset
    
    # mandamos el comando para que el target realice el cálculo 
    target.flush()
    scope.arm()
    target.simpleserial_write("g", bytearray([]))
    
    # esperamos respuesta 
    ret = scope.capture()
    val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)
    
    # Caso 1: Timeout de trigger (el target no arrancó correctamente)
    if ret:
        print('Timeout - no trigger')
        reboot_flush()
        return None
    
    # Caso 2: Cuelgue por glitch (la respuesta serie no es válida)
    if not val['valid']:
        reboot_flush()
        return None
        
    # Caso 3: Éxito en la comunicación, devolvemos el valor entero
    return struct.unpack("<I", val['payload'])[0]


Prueba a insertar glitches de forma que obtengas resultados válidos y algún reseteo, variando los valores de offset y repeat:

In [ ]:
off = 1
rep = 1
val = insert_glitch(off, rep)
if val is None:
    print("Se ha producido un RESET\n")
else: 
    print(val)

# Exploración de los parámetros del ataque

Tienes que hacer una búsqueda de cuáles son los parámetros (offset y repeat) adecuados para introducir un glitch en la alimentación del target de forma que no se cuelgue ni resetee y que devuelva un valor distinto a 2500, indicando que hemos conseguido que alguna de las instrucciones no sea ejecutada y el hardware no lo detecte.

Algunos consejos para encontrar fallos (*glitches*) efectivos:

1. Se trata de una línea de VCC que estamos cortocircuitando, por lo que habrá elementos jugando en nuestra contra (como la capacitancia del circuito). Si el *glitch* es demasiado corto, es posible que no tenga ningún efecto.
2. Del mismo modo, si el *glitch* es demasiado largo, el objetivo siempre se colgará (*crash*). Normalmente existe una estrecha franja o margen donde se consigue alterar el funcionamiento del objetivo sin llegar a bloquearlo por completo de forma sistemática.
3. Ten paciencia. La inyección de fallos puede ser algo inconsistente, ¡así que no te desanimes si tardas un poco en ver los primeros resultados positivos!

Por si quieres hacer la búsqueda automática y de forma gráfica, en el siguiente bloque de código aparece un código que recorre un rango dado para dos valores y pinta en la gráfica marcas del resultado de la exploración. La idea es que pintes un aspa roja en la coordenada de offset,repeat donde se ha producido un reset y un + verde en la coordenada donde se ha conseguido un valor distinto de 2500. No pintes nada para el caso más común (2500).

Antes de hacer la exploración, configuramos el controlador de "Glitches" (`GlitchController`). Esta clase es una herramienta gráfica de ChipWhisperer que facilita automatizar ataques. Se configuran tres categorías para los resultados:
* `normal`: No pasó nada raro.
* `reset`: El target se colgó.
* `success`: ¡Se ha logrado alterar el contador! 

También dibuja la gráfica interactiva donde irán apareciendo marcas en tiempo real a medida que se inyectan fallos (los resets los representará con un aspa roja y cuando se consiga un ataque exitoso con un + verde).

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["repeat", "ext_offset"])
gc.display_stats()
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

El siguiente código muestra cómo quedarían en la gráfica reseteos y glitches exitosos (lo hace en base a un número aleatorio) en función del offset y repeat correspondiente. 

Ejecuta el código para ver cómo queda y luego modifícalo para llevar a cabo glitches con cada uno de los valores de los parámetros y pinta el resultado de forma similar al del ejemplo (elimina la generación del número aleatorio y sustitúyela por la generación del glitch, interpreta el resultado y píntalo en la gráfica). 

In [ ]:
import random

g_step = 1

gc.set_global_step(g_step)
gc.set_range("repeat", 1, 2)       # definición del espacio de búsqueda para parámetro repeat 
gc.set_range("ext_offset", 1, 20)  # definición del espacio de búsqueda para parámetro ext_offset  
scope.glitch.repeat = 0

reboot_flush()
sample_size = 3                   # número de veces que repetimos un mismo glitch para mejorar consistencia  
for glitch_setting in gc.glitch_values():
    repeat, ext_offset = glitch_setting
    successes = 0                 
    resets = 0
    
    for i in range(sample_size):
        glitch_result = random.randint(0,2) # Aquí va tu generación de glitch para parámetros repeat y ext_offset 
        
        if glitch_result == 0:   # aquí deberás comprobar si se ha producido reset  
            gc.add("reset")
            resets +=1
        elif glitch_result == 1: # aquí si ha habido éxito al insertar el glitch  
            gc.add("success")
            successes += 1
            print(f"Glitch con éxito! Contador={glitch_result}")
        else:                    # aquí cuando se ha ejecutado normalmente el código en el target  
            gc.add("normal")
        
    if successes > 0:
        print("successes = {}, resets = {}, repeat = {}, ext_offset = {}".format(
            successes, resets, repeat, ext_offset))

print("Done glitching")

In [ ]:
gc.plot_2d(alpha=False)

# Glitching de Tensión para Eludir la Verificación de Contraseña 

Hemos visto cómo se puede utilizar el glitching de tensión para corromper cálculos. Ahora vamos a ver si también puede emplearse para eludir una comprobación de contraseña (*password check*).

Para ello, el target va a ejecutar una comprobación de contraseña muy simple, cuyo código fuente puedes ver en el mismo firmware que ya está cargado (`simpleserial-glich.c`) en la función `password()` que es asociada al comando `p` en este caso. 

La función tiene este aspecto:

```C
    #if SS_VER == SS_VER_2_1
    uint8_t password(uint8_t cmd, uint8_t scmd, uint8_t len, uint8_t* pw)
    #else
    uint8_t password(uint8_t* pw, uint8_t len)
    #endif
    {
        char passwd[] = "touch";
        char passok = 1;
        int cnt;
    
        trigger_high();
    
        //Simple test - doesn't check for too-long password!
        for(cnt = 0; cnt < 5; cnt++){
            if (pw[cnt] != passwd[cnt]){
                passok = 0;
            }
        }
    
        trigger_low();
    
        simpleserial_put('r', 1, (uint8_t*)&passok);
        return 0x00;
    }
```

Cuando se le envía el comando `p` al target junto con una cadena de 5 caracteres, se invoca esa función que chequea uno a uno los 5 caracteres enviados para ver si coinciden con los del password, que en este caso es "touch". 

La función envía de vuelta (por el puerto serie) una `r` y un byte que vale 0 si el password es inválido y un 1 si es válido. 

En esta parte de la práctica vamos a intentar introducir un glitch que genere un fallo de ejecución que haga que se devuelva un 1 aunque el password enviado sea inválido. 

Antes de realizar el ataque, vamos a ver el código ensamblador de esta función que realmente se ejecuta en el target, una vez compilado el firmware:

```ASM
    080004b0 <password>:
       ... (inicializaciones y carga de "touch" en la pila)
     80004bc:  movs  r6, #1
     80004c4:  strb  r6, [sp, #7]          ; passok = 1 (se guarda inicialmente en la pila)
    
     80004c6:  bl    trigger_high          ; <-- TRIGGER EN ALTO
     80004ca:  movs  r3, #0                ; r3 = 0 (r3 actúa como bandera de "error encontrado")
    
    ; ====== BUCLE DE COMPARACIÓN (cnt de 0 a 4) ======
     80004cc:  movs  r2, r3                ; r2 = cnt = 0
     80004ce:  ldrb  r0, [r4, r2]          ; Carga pw[cnt]
     80004d0:  ldrb  r1, [r5, r2]          ; Carga passwd[cnt] ("touch")
     80004d2:  cmp   r0, r1                ; ¿Son iguales?
     80004d4:  beq.n 80004d8               ; Si son iguales, salta
     80004d6:  movs  r3, r6                ; Si son distintos: r3 = 1 (marca error)
     80004d8:  adds  r2, #1                ; cnt++
     80004da:  cmp   r2, #5                ; ¿cnt == 5?
     80004dc:  bne.n 80004ce               ; Si cnt < 5, repite el bucle
    
    ; ====== DECISIÓN FINAL TRAS TERMINAR EL BUCLE ======
     80004de:  cmp   r3, #0                ; ¿Hubo algún error en todo el bucle?
     80004e0:  beq.n 80004e8               ; SI r3 == 0 (sin errores): SALTA A TRIGGER_LOW
     80004e2:  movs  r3, #0                ; Si hubo error: prepara un 0
     80004e4:  mov   r2, sp
     80004e6:  strb  r3, [r2, #7]          ; passok = 0 (¡SOLO SE ESCRIBE AQUÍ, UNA SOLA VEZ!)
    
     80004e8:  bl    trigger_low           ; <-- TRIGGER EN BAJO
     80004ec:  ... (envía passok por simpleserial_put)

```

Si analizamos el código ensamblador, vemos que al comienzo se escribe en memoria el valor 1 para `passok` (`strb  r6, [sp, #7] `). Durante el bucle, si se detecta un carácter erróneo en el password, se escribe un 1 en el registro r3 (`movs r3, r6`). Fuera del bucle, si r3 vale 1 se escribe 0 en `passok` (`strb  r3, [r2, #7]`) indicando fallo en el chequeo del password.

Por tanto, existen varios puntos en los que si insertamos el glitch y no se ejecuta correctamente la instrucción correspondiente, la función devolverá un 1 (`passok`=1) para un password erróneo. 

Por ejemplo, si no se ejecuta la instrucción `movs  r3, #0` del final, que pone un 0 en r3 para escribirlo luego en `passok`, r3 seguirá con el valor 1 que indica password correcto. 

Por otro lado, si el glitch afecta a la escritura en memoria (`strb  r3, [r2, #7]`) y evita que se ejecute, el 1 cargado al comienzo de la función en `passok` se mantendrá y no será reescrito por un 0. 

También podría afectar a la propia comparación (`cmp r3,#0`) o al salto (`beq.n 80004e8`) y hacer que saltara a `trigger_low` sin hacer el store del 0 en `passok`.

Vamos a realizar ahora una prueba del funcionamiento del código que chequea el password, primero con el password correcto y luego con uno incorrecto. 

Si te fijas es muy similar a cómo invocábamos el código que calculaba el 2500 en las pruebas anteriores, solo que ahora se envía el comando  `p` en vez del `g` con la función `target.simpleserial_write()` junto con un array de bytes con los códigos ASCII de los 5 caracteres que representan el password. La respuesta ahora, si es válida, en el `payload` viene un byte a 0 si el password es incorrecto o a 1 si es correcto. 

In [ ]:
reboot_flush()
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68])     # Código ASCII de "touch"  
target.simpleserial_write('p', pw)                 # Enviamos el password al target 
scope.capture()
val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10) # Recogemos respuesta
valid = val['valid']                               
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

In [ ]:
reboot_flush()
pw = bytearray([0x00, 0x00, 0x00, 0x00, 0x00])     # Enviamos 5 caracteres inválidos
target.simpleserial_write('p', pw)
scope.capture()
val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

Ahora, al igual que hicimos en el ataque anterior, vamos a definir una función que inserte el glitch pero ahora enviando el password:

In [ ]:
def insert_glitch_password(offset, repeat):
    """
    Inyecta un glitch en la comprobación del password y devuelve:
        - false: si el password es incorrecto
        - true: si el password es correcto, es decir, ha tenido éxito el ataque.
        - None: si el target se colgó o se reinició.
    """
    # configuramos los parámetros del glitch  
    scope.glitch.ext_offset = ext_offset
    scope.glitch.repeat = repeat

    # mandamos el comando para que el target chequee el password 
    target.flush()
    scope.arm()
    target.simpleserial_write('p', bytearray([0]*5))

    # esperamos respuesta 
    ret = scope.capture()
    if ret:
        print('Timeout - no trigger')
        reboot_flush()
        return None
        
    val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)
    if not val['valid']:
        reboot_flush()
        return None
        
    # Devuelve True si aceptó la contraseña (1), False si la rechazó (0)
    return val['payload'][0] == 1

Prueba a insertar unos cuantos glitches que den password inválido y algún reseteo, variando los valores de offset y repeat:

In [ ]:
off = 1
rep = 1
val = insert_glitch_password(off, rep)
if val is None:
    print("Se ha producido un RESET\n")
else: 
    print("Password correcto" if val else "Password incorrecto")

Ahora, al igual que en el ataque anterior, tienes que hacer una búsqueda de los parámetros offset y repeat que provoquen un ataque exitoso, en el que se obtenga un chequeo de password "positivo" sin haber introducido el password correcto. 

Muestra los resultados de forma gráfica como antes, con un aspa roja para los reseteos y un + verde para cuando consigas saltarte la protección por password (en este ataque puedes terminar de buscar en cuanto hayas roto la protección ;)

**Idea**: quizás en vez de 3 repeticiones de glitch por cada par de offset y repeat, podrías utilizar 50 para que sea un ataque más robusto ;)

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["repeat", "ext_offset"])
gc.display_stats()
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Realiza el escaneo de los parámetros para romper el chequeo del password (recuerda, en cuanto tengas un éxito para el escaneo):

In [ ]:
gc.plot_2d(alpha=False)

Para terminar, cerramos las conexiones por USB con el hardware de forma segura para evitar que los puertos se queden bloqueados o "zombis" en el sistema operativo del ordenador. Siempre se debe ejecutar al terminar de usar el hardware.

In [ ]:
scope.dis()
target.dis()

---
**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir de los notebooks originales [*Fault 2_1 - Introduction to Voltage Glitching.ipynb*](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/fault101/Fault%202_1%20-%20Introduction%20to%20Voltage%20Glitching.ipynb) y [*Fault 2_2 - Voltage Glitching to Bypass Password.ipynb*](https://github.com/newaetech/chipwhisperer-jupyter/blob/dee4ddd71fa534457b97df6880ad4fecbc77bb94/courses/fault101/Fault%202_2%20-%20Voltage%20Glitching%20to%20Bypass%20Password.ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).